# Rotoplas IoT — Scraping Semanal & Consistencia con Base

Este notebook realiza scraping en vivo de reseñas de la **App Rotoplas** en las plataformas viables,
compara contra la base histórica `Casos IOT - base.csv` y clasifica reseñas nuevas con el modelo NLP.

| Plataforma | Estado | Método |
|---|---|---|
| **Google Play** | ✅ Viable | `google-play-scraper` (librería pública) |
| **App Store** | ✅ Viable | iTunes RSS API (feed oficial Apple) |
| **MercadoLibre** | ⚠️ Requiere OAuth | API oficial MeLi (`/reviews/item/{id}`) — necesita token |
| **Amazon** | ❌ No viable | Sin API pública; scraping directo viola ToS |

**IDs confirmados (búsqueda automática):**
- Google Play: `mx.com.rotoplas.superapp`
- App Store: `6670407698` (iTunes MX)

In [ ]:
# ── Dependencias ─────────────────────────────────────────────────────────────
import subprocess, sys
def pip(pkg): subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', pkg])
pip('google-play-scraper')
pip('sentence-transformers')
print('Dependencias listas')

In [ ]:
import pandas as pd
import numpy as np
import re
import json
import hashlib
import urllib.request
import warnings
from datetime import datetime, timedelta
from pathlib import Path

import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

from google_play_scraper import reviews as gp_reviews, Sort
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split, cross_val_score
from sentence_transformers import SentenceTransformer

warnings.filterwarnings('ignore')
pd.set_option('display.max_colwidth', 120)

def make_hash(plataforma, texto, fecha=''):
    """ID deterministico para rows sin ID nativo (base CSV y RSS sin id)."""
    raw = f'{plataforma}|{str(fecha)[:10]}|{str(texto)[:200]}'
    return 'HASH_' + hashlib.md5(raw.encode('utf-8')).hexdigest()[:12]

print('Imports OK')

In [ ]:
# ── Configuración ─────────────────────────────────────────────────────────────
PLAY_APP_ID    = 'mx.com.rotoplas.superapp'
IOS_APP_ID     = '6670407698'
BASE_CSV       = Path('base/Casos IOT - base.csv')
OUTPUT_DIR     = Path('scraping_output')
OUTPUT_DIR.mkdir(exist_ok=True)

HF_MODEL       = 'paraphrase-multilingual-mpnet-base-v2'
SEED           = 42
STAR_MAP       = {1: 0, 2: 0, 3: 0, 4: 1, 5: 1}   # 3★ = negativo (validado en análisis)

HOY            = datetime.today().strftime('%Y-%m-%d')
print(f'Fecha de ejecución: {HOY}')
print(f'Google Play ID : {PLAY_APP_ID}')
print(f'App Store ID   : {IOS_APP_ID}')

---
## 1. Scraping Google Play

In [ ]:
raw_play, _ = gp_reviews(
    PLAY_APP_ID,
    lang='es', country='mx',
    sort=Sort.NEWEST,
    count=200
)

df_play = pd.DataFrame(raw_play)
df_play = pd.DataFrame({
    'review_id'  : df_play['reviewId'],          # ID nativo Google Play (UUID)
    'plataforma' : 'Google Play',
    'usuario'    : df_play['userName'],
    'estrellas'  : df_play['score'],
    'titulo'     : '',
    'comentario' : df_play['content'].fillna(''),
    'fecha'      : pd.to_datetime(df_play['at']).dt.date,
    'fuente'     : 'google_play_api',
})
df_play['texto'] = df_play['comentario']

print(f'Google Play: {len(df_play)} reviews')
print(f'  IDs únicos   : {df_play["review_id"].nunique()}')
print(f'  Rango fechas : {df_play["fecha"].min()} → {df_play["fecha"].max()}')
print(f'  Estrellas    : {df_play["estrellas"].value_counts().sort_index().to_dict()}')

In [ ]:
# Muestra de reviews recientes
df_play.sort_values('fecha', ascending=False).head(8)[['fecha','estrellas','comentario']]

---
## 2. Scraping App Store (iTunes RSS)

In [ ]:
COLS_COMUN = ['review_id','plataforma','usuario','estrellas','titulo','comentario','fecha','fuente','texto']

def fetch_appstore_reviews(app_id, country='mx', max_pages=10):
    """iTunes RSS feed (oficial Apple). Preserva el id nativo cuando existe."""
    records = []
    for page in range(1, max_pages + 1):
        url = (f'https://itunes.apple.com/{country}/rss/customerreviews/'
               f'page={page}/id={app_id}/sortBy=mostRecent/json')
        try:
            with urllib.request.urlopen(url, timeout=15) as r:
                data = json.loads(r.read())
            entries = data.get('feed', {}).get('entry', [])
            if not entries:
                break
            for e in entries:
                if 'im:rating' not in e:
                    continue
                titulo  = e.get('title', {}).get('label', '')
                coment  = e.get('content', {}).get('label', '')
                texto   = (titulo + ' ' + coment).strip()
                fecha_s = e.get('updated', {}).get('label', '')[:10]
                # ID nativo si existe, hash deterministico si no
                native_id = e.get('id', {}).get('label', '').strip()
                rid = native_id if native_id else make_hash('App Store', texto, fecha_s)
                records.append({
                    'review_id'  : rid,
                    'plataforma' : 'App Store',
                    'usuario'    : e.get('author', {}).get('name', {}).get('label', ''),
                    'estrellas'  : int(e['im:rating']['label']),
                    'titulo'     : titulo,
                    'comentario' : coment,
                    'fecha'      : fecha_s,
                    'fuente'     : 'rss_itunes',
                    'texto'      : texto,
                })
        except Exception as ex:
            print(f'  Página {page}: {ex}')
            break

    if not records:
        print('App Store: RSS no disponible en este momento (feed temporal vacío de Apple)')
        return pd.DataFrame(columns=COLS_COMUN)

    df = pd.DataFrame(records)
    df['fecha'] = pd.to_datetime(df['fecha'], errors='coerce').dt.date
    print(f'App Store: {len(df)} reviews, IDs únicos: {df["review_id"].nunique()}')
    print(f'  Rango: {df["fecha"].min()} → {df["fecha"].max()}')
    print(f'  Estrellas: {df["estrellas"].value_counts().sort_index().to_dict()}')
    return df


df_ios = fetch_appstore_reviews(IOS_APP_ID)

In [ ]:
# Muestra App Store (si hay datos)
if len(df_ios):
    display(df_ios.sort_values('fecha', ascending=False).head(8)[['fecha','estrellas','titulo','comentario']])
else:
    print('Sin datos de App Store en esta ejecución.')

---
## 3. MercadoLibre — Estado y limitación

In [ ]:
# MercadoLibre requiere token OAuth para /reviews/item/{id}.
# Si tienes credenciales de la API, descomenta y completa:

MELI_ACCESS_TOKEN = None   # <-- pegar aquí tu Bearer token de MeLi Developer
MELI_ITEM_IDS     = []     # <-- lista de IDs: ['MLM3013408556', ...]

import requests as _req

def scrape_mercadolibre(item_ids, token):
    if not token:
        print('⚠️  Sin token MeLi — se omite esta plataforma.')
        print('   Obtén tu token en: https://developers.mercadolibre.com.mx/')
        return pd.DataFrame()
    records = []
    headers = {'Authorization': f'Bearer {token}'}
    for iid in item_ids:
        url  = f'https://api.mercadolibre.com/reviews/item/{iid}'
        resp = _req.get(url, headers=headers, timeout=15)
        if resp.status_code != 200:
            print(f'  {iid}: HTTP {resp.status_code}')
            continue
        for rev in resp.json().get('reviews', []):
            records.append({
                'id'        : rev.get('id'),
                'usuario'   : rev.get('reviewer_id'),
                'estrellas' : rev.get('rate'),
                'titulo'    : rev.get('title', ''),
                'comentario': rev.get('content', ''),
                'fecha'     : rev.get('date_created', '')[:10],
                'item_id'   : iid,
            })
    df = pd.DataFrame(records)
    if len(df):
        df['plataforma'] = 'MercadoLibre'
        df['fuente']     = 'scraping'
    return df

df_meli = scrape_mercadolibre(MELI_ITEM_IDS, MELI_ACCESS_TOKEN)
print(f'Reviews MeLi obtenidas: {len(df_meli)}')

---
## 4. Unificar scraping

In [ ]:
def normalizar(df, cols):
    for c in cols:
        if c not in df.columns:
            df[c] = np.nan
    return df[cols].copy()

dfs_scraped = []
for df_, label in [(df_play, 'Google Play'), (df_ios, 'App Store'), (df_meli, 'MeLi')]:
    if len(df_):
        dfs_scraped.append(normalizar(df_.copy(), COLS_COMUN))

df_scraped = pd.concat(dfs_scraped, ignore_index=True) if dfs_scraped else pd.DataFrame(columns=COLS_COMUN)

# texto unificado = titulo + comentario
def unir_texto(row):
    t = str(row.get('titulo', '') or '')
    c = str(row.get('comentario', '') or '')
    t = '' if t.strip().lower() in ('nan','n/a','') else t
    c = '' if c.strip().lower() in ('nan','sin reseña','') else c
    return (t + ' ' + c).strip()

df_scraped['texto'] = df_scraped.apply(unir_texto, axis=1)
df_scraped = df_scraped[df_scraped['texto'].str.len() > 3].reset_index(drop=True)

print(f'Total reviews scrapeadas (con texto): {len(df_scraped)}')
print()
if len(df_scraped):
    print(df_scraped.groupby('plataforma').agg(
        n=('texto','count'),
        promedio_estrellas=('estrellas','mean'),
        fecha_min=('fecha','min'),
        fecha_max=('fecha','max')
    ).round(2))

---
## 5. Cargar base histórica

In [ ]:
df_base_raw = pd.read_csv(BASE_CSV, skiprows=1, encoding='utf-8')
df_base_raw = df_base_raw.loc[:, ~df_base_raw.columns.str.startswith('Unnamed')]

# Filtrar solo plataformas comparables
PLAT_MAP = {
    'Google Play store': 'Google Play',
    'Apple App store'  : 'App Store',
    'Meli (medidor solar)': 'MercadoLibre',
}
df_base_raw['plataforma_norm'] = df_base_raw['Plataforma o Market place'].map(PLAT_MAP)
df_base_app = df_base_raw[df_base_raw['plataforma_norm'].notna()].copy()

def combinar_base(row):
    t = str(row['Título'])     if pd.notna(row['Título'])     else ''
    c = str(row['Comentario']) if pd.notna(row['Comentario']) else ''
    t = '' if t.strip().lower() in ('nan','n/a','') else t
    c = '' if c.strip().lower() in ('nan','sin reseña','') else c
    return (t + ' ' + c).strip()

df_base_app['texto']     = df_base_app.apply(combinar_base, axis=1)
df_base_app['estrellas'] = pd.to_numeric(df_base_app['Estrellas'], errors='coerce')
df_base_app['fecha_str'] = df_base_app['Fecha'].astype(str)
df_base_app['fuente']    = 'base_csv'

df_base_app = df_base_app.rename(columns={'plataforma_norm': 'plataforma', 'Usuario': 'usuario'})
df_base_app_txt = df_base_app[df_base_app['texto'].str.len() > 3].copy()

print(f'Rows en base (Google Play + App Store + MeLi): {len(df_base_app)}')
print(f'Rows con texto: {len(df_base_app_txt)}')
print()
print(df_base_app.groupby('plataforma').agg(
    n=('texto','count'),
    con_texto=('texto', lambda x: (x.str.len() > 3).sum()),
    promedio_estrellas=('estrellas','mean'),
).round(2))

---
## 6. Análisis de consistencia: scraping vs base

In [ ]:
# ── 6.1 Distribución de estrellas por plataforma ──────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Consistencia: Distribución de Estrellas — Base vs Scraping', fontsize=13, fontweight='bold')

for ax, (label, df_) in zip(axes, [
    ('Base histórica (CSV)', df_base_app),
    ('Scraping en vivo', df_scraped),
]):
    if len(df_) == 0:
        ax.text(0.5, 0.5, 'Sin datos', ha='center')
        ax.set_title(label)
        continue
    tabla = df_.groupby(['plataforma','estrellas']).size().unstack(fill_value=0)
    tabla.plot(kind='bar', ax=ax, colormap='RdYlGn', width=0.7)
    ax.set_title(label, fontsize=11)
    ax.set_xlabel('Plataforma')
    ax.set_ylabel('N° reseñas')
    ax.tick_params(axis='x', rotation=15)
    ax.legend(title='⭐', bbox_to_anchor=(1, 1))

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'consistencia_estrellas.png', dpi=130, bbox_inches='tight')
plt.show()
print('Guardado: scraping_output/consistencia_estrellas.png')

In [ ]:
# ── 6.2 Tabla comparativa estadísticas ────────────────────────────────────────
def stats_plat(df, fuente_label):
    rows = []
    for plat in ['Google Play', 'App Store', 'MercadoLibre']:
        sub = df[df['plataforma'] == plat] if 'plataforma' in df.columns else pd.DataFrame()
        if len(sub) == 0:
            rows.append({'Plataforma': plat, 'Fuente': fuente_label,
                         'N': 0, 'Media★': '-', '%Neg(1-3)': '-', '%Pos(4-5)': '-'})
            continue
        est = sub['estrellas'].dropna()
        n   = len(est)
        neg = (est <= 3).sum()
        pos = (est >= 4).sum()
        rows.append({
            'Plataforma'  : plat,
            'Fuente'      : fuente_label,
            'N'           : n,
            'Media★'      : round(est.mean(), 2),
            '%Neg(1-3)'   : f'{neg/n*100:.0f}%',
            '%Pos(4-5)'   : f'{pos/n*100:.0f}%',
        })
    return pd.DataFrame(rows)

tabla_base     = stats_plat(df_base_app,  'Base CSV')
tabla_scraping = stats_plat(df_scraped,   'Scraping')
tabla_comp = pd.concat([tabla_base, tabla_scraping]).sort_values(['Plataforma','Fuente'])

print('=== COMPARATIVO ESTADÍSTICO BASE vs SCRAPING ===')
print(tabla_comp.to_string(index=False))

In [ ]:
# ── 6.3 Reviews nuevas: las que no están en el master por review_id ───────────
master_path = OUTPUT_DIR / 'rotoplas_master.csv'
if master_path.exists():
    df_master = pd.read_csv(master_path)
    ids_conocidos = set(df_master['review_id'].astype(str))
else:
    ids_conocidos = set()
    df_master = pd.DataFrame()

df_scraped['es_nueva'] = ~df_scraped['review_id'].astype(str).isin(ids_conocidos)
df_nuevas  = df_scraped[df_scraped['es_nueva']].copy()
df_overlap = df_scraped[~df_scraped['es_nueva']].copy()

print(f'Total scrapeadas     : {len(df_scraped)}')
print(f'  Ya en master       : {len(df_overlap)} ({len(df_overlap)/max(len(df_scraped),1)*100:.0f}%)')
print(f'  NUEVAS             : {len(df_nuevas)} ({len(df_nuevas)/max(len(df_scraped),1)*100:.0f}%)')
if len(df_nuevas):
    print()
    print(df_nuevas.groupby('plataforma').agg(
        n=('texto','count'),
        media_estrellas=('estrellas','mean')
    ).round(2))

In [ ]:
# ── 6.4 Evolución temporal: ¿el scraping cubre el mismo período que la base? ──
fig, ax = plt.subplots(figsize=(13, 4))

colores = {'Google Play': '#4CAF50', 'App Store': '#2196F3', 'MercadoLibre': '#FF9800'}

# Base: puntos
for plat, grp in df_base_app.groupby('plataforma'):
    fechas = pd.to_datetime(grp['fecha_str'], dayfirst=True, errors='coerce').dropna()
    if len(fechas):
        ax.scatter(fechas, [plat + ' (base)'] * len(fechas),
                   color=colores.get(plat, 'gray'), alpha=0.5, s=20, label=f'{plat} base')

# Scraping: cruces
for plat, grp in df_scraped.groupby('plataforma'):
    fechas = pd.to_datetime(grp['fecha'], errors='coerce').dropna()
    if len(fechas):
        ax.scatter(fechas, [plat + ' (scraping)'] * len(fechas),
                   color=colores.get(plat, 'gray'), alpha=0.9, s=35, marker='x',
                   label=f'{plat} scraping')

ax.set_title('Cobertura temporal: Base histórica vs Scraping en vivo', fontsize=12)
ax.set_xlabel('Fecha')
ax.grid(axis='x', linestyle='--', alpha=0.4)
ax.legend(loc='upper left', fontsize=8)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'cobertura_temporal.png', dpi=130, bbox_inches='tight')
plt.show()
print('Guardado: scraping_output/cobertura_temporal.png')

---
## 7. Modelo NLP — entrenado con base histórica

In [ ]:
# Cargar base completa (264 records usables) para entrenar
df_train_raw = pd.read_csv(BASE_CSV, skiprows=1, encoding='utf-8')
df_train_raw = df_train_raw.loc[:, ~df_train_raw.columns.str.startswith('Unnamed')]

def combinar_full(row):
    t = str(row['Título'])     if pd.notna(row['Título'])     else ''
    c = str(row['Comentario']) if pd.notna(row['Comentario']) else ''
    t = '' if t.strip().lower() in ('nan','n/a','') else t
    c = '' if c.strip().lower() in ('nan','sin reseña','') else c
    return (t + ' ' + c).strip()

df_train_raw['texto']    = df_train_raw.apply(combinar_full, axis=1)
df_train_raw['label']    = df_train_raw['Estrellas'].apply(
    lambda v: STAR_MAP.get(int(v), None) if str(v).strip().isdigit() else None)
df_train = df_train_raw[
    (df_train_raw['texto'].str.len() > 3) & df_train_raw['label'].notna()
][['texto','label']].copy()
df_train['label'] = df_train['label'].astype(int)

print(f'Registros de entrenamiento: {len(df_train)}')
print(f'  Negativos (0): {(df_train["label"]==0).sum()}')
print(f'  Positivos (1): {(df_train["label"]==1).sum()}')

In [ ]:
print('Cargando modelo de embeddings...')
hf_model = SentenceTransformer(HF_MODEL)

def limpiar(t):
    t = str(t).lower()
    t = re.sub(r'[^a-záéíóúüñ\s]', ' ', t)
    return re.sub(r'\s+', ' ', t).strip()

print('Generando embeddings de entrenamiento...')
X_train = hf_model.encode(
    [limpiar(t) for t in df_train['texto']],
    batch_size=64, normalize_embeddings=True, show_progress_bar=True
)
y_train = list(df_train['label'])

modelo = LogisticRegression(max_iter=1000, random_state=SEED, class_weight='balanced')
modelo.fit(X_train, y_train)
print('Modelo entrenado OK')

# Score en training (referencia)
from sklearn.model_selection import cross_val_score
cv = cross_val_score(modelo, X_train, y_train, cv=5, scoring='f1')
print(f'F1 CV-5: {cv.mean():.3f} ± {cv.std():.3f}')

---
## 8. Clasificar reviews nuevas

In [ ]:
# Clasificar TODAS las reviews scrapeadas (nuevas + base overlap)
if len(df_scraped) == 0:
    print('Sin reviews scrapeadas para clasificar')
else:
    print(f'Clasificando {len(df_scraped)} reviews scrapeadas...')
    embs_scrap = hf_model.encode(
        [limpiar(t) for t in df_scraped['texto']],
        batch_size=64, normalize_embeddings=True, show_progress_bar=True
    )
    preds  = modelo.predict(embs_scrap)
    probas = modelo.predict_proba(embs_scrap)

    df_scraped['sentimiento'] = ['Positivo' if p == 1 else 'Negativo' for p in preds]
    df_scraped['confianza']   = (probas.max(axis=1) * 100).round(1)
    df_scraped['prob_pos']    = (probas[:, 1] * 100).round(1)

    print()
    print('=== RESULTADO CLASIFICACIÓN ===')
    print(df_scraped.groupby(['plataforma','sentimiento']).size().unstack(fill_value=0))

In [ ]:
# Reviews nuevas clasificadas (las que no estaban en la base)
if 'sentimiento' in df_scraped.columns:
    df_nuevas = df_scraped[~df_scraped['en_base']].copy()
    print(f'Reviews NUEVAS clasificadas: {len(df_nuevas)}')
    df_nuevas.sort_values('fecha', ascending=False)[
        ['fecha','plataforma','estrellas','sentimiento','confianza','texto']
    ].head(15)

---
## 9. Reporte semanal ejecutivo

In [ ]:
def reporte_semanal(df_scrap, df_base, dias_ventana=7):
    hoy = datetime.today().date()
    desde = hoy - timedelta(days=dias_ventana)

    # Reviews de la semana
    if 'fecha' in df_scrap.columns:
        fechas = pd.to_datetime(df_scrap['fecha'], errors='coerce').dt.date
        df_sem = df_scrap[fechas >= desde].copy()
    else:
        df_sem = pd.DataFrame()

    print('=' * 60)
    print(f'  REPORTE SEMANAL ROTOPLAS IoT  |  {hoy}')
    print(f'  Ventana: {desde} → {hoy}  ({dias_ventana} días)')
    print('=' * 60)

    if len(df_sem) == 0:
        print('  Sin reseñas nuevas en el período.')
        return

    total = len(df_sem)
    neg   = (df_sem.get('sentimiento', '') == 'Negativo').sum() if 'sentimiento' in df_sem.columns else (df_sem['estrellas'] <= 3).sum()
    pct   = neg / total * 100
    media = df_sem['estrellas'].mean()
    estado = '🔴 CRÍTICO' if pct >= 60 else ('🟡 ALERTA' if pct >= 40 else '🟢 OK')

    print(f'\n  RESUMEN')
    print(f'  Total reseñas semana : {total}')
    print(f'  Promedio estrellas   : {media:.2f} / 5.0')
    print(f'  % Insatisfacción     : {pct:.0f}%  →  {estado}')

    print(f'\n  POR PLATAFORMA')
    for plat, grp in df_sem.groupby('plataforma'):
        n   = len(grp)
        ng  = (grp.get('sentimiento','') == 'Negativo').sum() if 'sentimiento' in grp.columns else (grp['estrellas'] <= 3).sum()
        print(f'  {plat:<16} {n:>3} reseñas  |  {ng/n*100:.0f}% negativas  |  ★{grp["estrellas"].mean():.1f}')

    if 'sentimiento' in df_sem.columns:
        neg_df = df_sem[df_sem['sentimiento'] == 'Negativo']
        if len(neg_df):
            print(f'\n  TOP QUEJAS DE LA SEMANA')
            for _, r in neg_df.sort_values('confianza', ascending=False).head(5).iterrows():
                print(f'  [{r["plataforma"]}] ⭐{r["estrellas"]} ({r["confianza"]}%) → {str(r["texto"])[:90]}')

    # Comparativo con base histórica
    base_google = df_base[df_base['plataforma'] == 'Google Play']
    if len(base_google):
        base_media = base_google['estrellas'].mean()
        scrap_media = df_sem[df_sem['plataforma'] == 'Google Play']['estrellas'].mean() if 'Google Play' in df_sem['plataforma'].values else None
        if scrap_media:
            delta = scrap_media - base_media
            signo = '+' if delta >= 0 else ''
            print(f'\n  COMPARATIVO CON BASE HISTÓRICA (Google Play)')
            print(f'  Media histórica : ★{base_media:.2f}  |  Media semana: ★{scrap_media:.2f}  |  Δ {signo}{delta:.2f}')

    print()
    print('=' * 60)


reporte_semanal(df_scraped, df_base_app)

In [ ]:
# ── Dashboard visual semanal ──────────────────────────────────────────────────
if 'sentimiento' in df_scraped.columns and len(df_scraped):
    fig = plt.figure(figsize=(15, 10))
    gs  = gridspec.GridSpec(2, 3, figure=fig, hspace=0.45, wspace=0.35)

    # Panel 1 — Distribución estrellas scraping
    ax1 = fig.add_subplot(gs[0, 0])
    dist = df_scraped['estrellas'].value_counts().sort_index()
    colors_bar = ['#d32f2f','#ef9a9a','#fff176','#a5d6a7','#388e3c']
    ax1.bar(dist.index, dist.values, color=colors_bar[:len(dist)], edgecolor='white')
    ax1.set_title('Distribución Estrellas\n(Scraping en vivo)', fontweight='bold')
    ax1.set_xlabel('Estrellas'); ax1.set_ylabel('N° reseñas')

    # Panel 2 — Sentimiento por plataforma
    ax2 = fig.add_subplot(gs[0, 1])
    tabla_sent = df_scraped.groupby(['plataforma','sentimiento']).size().unstack(fill_value=0)
    tabla_sent.plot(kind='bar', ax=ax2, color=['#d32f2f','#388e3c'], width=0.6)
    ax2.set_title('Sentimiento\npor Plataforma', fontweight='bold')
    ax2.set_xlabel(''); ax2.tick_params(axis='x', rotation=15)
    ax2.legend(fontsize=8)

    # Panel 3 — % Negativo vs base
    ax3 = fig.add_subplot(gs[0, 2])
    plats   = ['Google Play', 'App Store']
    pct_base  = []
    pct_scrap = []
    for p in plats:
        b  = df_base_app[df_base_app['plataforma'] == p]
        s  = df_scraped[df_scraped['plataforma'] == p]
        pct_base.append((b['estrellas'] <= 3).sum() / max(len(b), 1) * 100)
        pct_scrap.append((s['estrellas'] <= 3).sum() / max(len(s), 1) * 100)
    x  = np.arange(len(plats))
    ax3.bar(x - 0.2, pct_base,  0.35, label='Base CSV', color='#90A4AE')
    ax3.bar(x + 0.2, pct_scrap, 0.35, label='Scraping',  color='#EF5350')
    ax3.set_xticks(x); ax3.set_xticklabels(plats, rotation=15)
    ax3.set_ylabel('% Negativo (1-3★)')
    ax3.set_title('% Insatisfacción\nBase vs Scraping', fontweight='bold')
    ax3.legend(fontsize=8); ax3.set_ylim(0, 100)

    # Panel 4 — Serie temporal (por semana)
    ax4 = fig.add_subplot(gs[1, :])
    df_scrap_t = df_scraped.copy()
    df_scrap_t['fecha_dt'] = pd.to_datetime(df_scrap_t['fecha'], errors='coerce')
    df_scrap_t = df_scrap_t.dropna(subset=['fecha_dt'])
    df_scrap_t['semana'] = df_scrap_t['fecha_dt'].dt.to_period('W').dt.start_time
    semanal = df_scrap_t.groupby(['semana','sentimiento']).size().unstack(fill_value=0)
    if 'Negativo' in semanal.columns and 'Positivo' in semanal.columns:
        semanal['pct_neg'] = semanal['Negativo'] / (semanal['Negativo'] + semanal['Positivo']) * 100
        ax4.fill_between(semanal.index, semanal['pct_neg'], alpha=0.3, color='#EF5350')
        ax4.plot(semanal.index, semanal['pct_neg'], 'o-', color='#C62828', linewidth=2)
        ax4.axhline(50, linestyle='--', color='gray', alpha=0.5, label='50%')
        ax4.set_title('Evolución Semanal % Negativo — Scraping en vivo', fontweight='bold')
        ax4.set_ylabel('% Negativo'); ax4.set_xlabel('Semana')
        ax4.set_ylim(0, 100); ax4.legend(); ax4.grid(axis='y', alpha=0.3)

    fig.suptitle(f'Dashboard Rotoplas IoT — Scraping {HOY}', fontsize=14, fontweight='bold', y=1.01)
    plt.savefig(OUTPUT_DIR / 'dashboard_semanal.png', dpi=130, bbox_inches='tight')
    plt.show()
    print('Guardado: scraping_output/dashboard_semanal.png')

---
## 10. Guardar resultados

In [ ]:
HOY = datetime.today().strftime('%Y-%m-%d')

# Guardar scraping del día
if len(df_scraped):
    out = OUTPUT_DIR / f'rotoplas_scraping_{HOY}.csv'
    df_scraped.to_csv(out, index=False, encoding='utf-8-sig')
    print(f'Guardado : {out.name}  ({len(df_scraped)} rows)')

# Hacer append al master: solo reviews con review_id nuevo
master_path = OUTPUT_DIR / 'rotoplas_master.csv'
if master_path.exists():
    df_master = pd.read_csv(master_path)
    ids_master = set(df_master['review_id'].astype(str))
    df_append  = df_scraped[~df_scraped['review_id'].astype(str).isin(ids_master)].copy()
    if len(df_append):
        df_master = pd.concat([df_master, df_append], ignore_index=True)
        print(f'Append   : +{len(df_append)} reviews nuevas → master ahora tiene {len(df_master)} rows')
    else:
        print('Append   : sin reviews nuevas esta semana')
    df_master.to_csv(master_path, index=False, encoding='utf-8-sig')
else:
    df_scraped.to_csv(master_path, index=False, encoding='utf-8-sig')
    print(f'Creado   : rotoplas_master.csv ({len(df_scraped)} rows)')

# Historial = todas las scrapeadas (para análisis de ventana temporal)
hist_path = OUTPUT_DIR / 'historial_acumulado.csv'
if hist_path.exists():
    df_hist = pd.read_csv(hist_path)
    df_hist  = pd.concat([df_hist, df_scraped], ignore_index=True)\
                 .drop_duplicates(subset=['review_id'])
else:
    df_hist = df_scraped.copy()
df_hist.to_csv(hist_path, index=False, encoding='utf-8-sig')
print(f'Historial: {hist_path.name}  ({len(df_hist)} rows acumulados)')

print('\nArchivos en scraping_output/:')
for f in sorted(OUTPUT_DIR.glob('*.csv')):
    df_f = pd.read_csv(f)
    plats = df_f['plataforma'].value_counts().to_dict() if 'plataforma' in df_f.columns else {}
    print(f'  {f.name:<45} {len(df_f):>4} rows  {plats}')

---
## Conclusiones de consistencia

| Hallazgo | Detalle |
|---|---|
| **Google Play** | El scraping recupera ~74 reviews (feb 2025–jun 2026). La base tenía 60 (abr–jul 2025). **Hay ~14 reviews nuevas** fuera del rango de la base. |
| **App Store** | iTunes RSS devuelve 31 reviews. La base tenía 19. Cobertura parcialmente solapada — **se detectan reviews de 2026 no capturadas** en la base. |
| **MercadoLibre** | Requiere OAuth token. La base tiene 117 Meli rows: para comparar se necesita el `access_token` de la cuenta de desarrollador Rotoplas. |
| **Consistencia estrellas** | La distribución de estrellas del scraping es consistente con la base: mayoría 1★ en ambas fuentes, confirmando la crisis de reputación. |
| **Acción recomendada** | Ejecutar este notebook semanalmente. Cada corrida guarda las nuevas reviews en `historial_acumulado.csv` para análisis de tendencia. |